# PARC2026 — Trajectory-group Leakage Gate V1

固定evalとtraining manifestの間で、**同じ非visual trajectory由来のepisode groupが跨っていないか**を検査します。

- `exact_group`: task + `observation.state` + `action` sequence
- `action_group`: task + `action` sequence only

`exact_group` がeval/trainの両方に存在する場合はLeakage GateをFAILにします。`action_group`だけの一致はcontrol-equivalent候補として別表示します。


## Self-contained preflight

`00` / `30` / `40` を先に実行していなくても、このNotebookだけでworkspace、repo、public proxy trajectory、Static Quality metrics、manifestを準備できます。GPUは不要です。


In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
print('python:', sys.version)
print('platform:', platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
ROOT = Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git', str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


## Datasetを準備

運営trajectoryがColabに無ければ、compact public `lerobot/libero_plus` v3の **meta + parquetのみ**を取得します。画像・動画はhashに使わないためdownloadしません。


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30','pyarrow>=16','pandas>=2'], check=True)
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

PUBLIC_DATASET = 'lerobot/libero_plus'
PUBLIC_ROOT = ROOT/'datasets'/'public_libero_plus_v3_quality'
ORGANIZER_ROOT = ROOT/'datasets'/'libero_combined_20hz'
configured = os.environ.get('PARC_DATASET_ROOT')

def ready(p):
    return (p/'meta'/'info.json').exists() and any(p.glob('data/**/*.parquet'))

if configured and ready(Path(configured)):
    DATASET_ROOT = Path(configured)
    DATASET_ID = os.environ.get('PARC_DATASET_ID','local/libero_combined_20hz')
    DATASET_REVISION = None
elif ready(ORGANIZER_ROOT):
    DATASET_ROOT = ORGANIZER_ROOT
    DATASET_ID = 'local/libero_combined_20hz'
    DATASET_REVISION = None
else:
    api = HfApi()
    info = api.dataset_info(PUBLIC_DATASET)
    DATASET_REVISION = info.sha
    files = api.list_repo_files(PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION)
    meta_files = sorted(f for f in files if f.startswith('meta/') and (f.endswith('.json') or f.endswith('.parquet')))
    data_files = sorted(f for f in files if f.startswith('data/') and f.endswith('.parquet'))
    for filename in [*meta_files, *data_files]:
        hf_hub_download(
            repo_id=PUBLIC_DATASET,
            repo_type='dataset',
            revision=DATASET_REVISION,
            filename=filename,
            local_dir=str(PUBLIC_ROOT),
        )
    DATASET_ROOT = PUBLIC_ROOT
    DATASET_ID = PUBLIC_DATASET

print('dataset:', DATASET_ID, '@', DATASET_REVISION)
print('root:', DATASET_ROOT)


## Static metrics / manifestを準備

既存の `30` / `40` 出力があれば再利用します。無ければ同じ設定で生成します。


In [ ]:
STATIC_OUT = ROOT/'outputs'/'static_quality_v1'
METRICS = STATIC_OUT/'episode_quality_metrics.csv'
if not METRICS.exists():
    print('static metrics not found; generating now...')
    subprocess.run([
        sys.executable, str(REPO/'tools/data/static_quality_analyzer.py'),
        '--root', str(DATASET_ROOT),
        '--out', str(STATIC_OUT),
        '--smooth-window','5',
        '--robust-z-threshold','5.0',
    ], check=True)

MANIFEST_OUT = ROOT/'outputs'/'dataset_ablation_manifests_v1'
if not (MANIFEST_OUT/'FIXED_EVAL_HOLDOUT.json').exists():
    print('ablation manifests not found; generating now...')
    cmd = [
        sys.executable, str(REPO/'tools/data/build_dataset_ablation_manifests.py'),
        '--metrics-csv', str(METRICS),
        '--out', str(MANIFEST_OUT),
        '--dataset-id', DATASET_ID,
        '--seed','20260830',
        '--eval-per-task','2',
    ]
    if DATASET_REVISION:
        cmd += ['--dataset-revision', DATASET_REVISION]
    subprocess.run(cmd, check=True)

print('metrics:', METRICS)
print('manifests:', MANIFEST_OUT)


## Trajectory fingerprint

float dtype差や極小serialization差でgroupが分裂しないよう、state/actionを小数6桁へ量子化してからSHA256化します。task indexとsequence lengthもhashに含めます。

これは**非visual trajectory一致の検出**であり、画像の同一性やtask successを判定するものではありません。


In [ ]:
LEAKAGE_OUT = ROOT/'outputs'/'trajectory_group_leakage_v1'
cmd = [
    sys.executable, str(REPO/'tools/data/check_trajectory_group_leakage.py'),
    '--root', str(DATASET_ROOT),
    '--manifests-dir', str(MANIFEST_OUT),
    '--out', str(LEAKAGE_OUT),
    '--metrics-csv', str(METRICS),
    '--round-decimals','6',
]
subprocess.run(cmd, check=True)
summary = json.loads((LEAKAGE_OUT/'trajectory_group_leakage_summary.json').read_text())
report = pd.read_csv(LEAKAGE_OUT/'manifest_leakage_report.csv')
display(report)


## Gate

- **PASS**: exact state+action trajectory groupがeval/trainを跨がない
- **FAIL**: 1 groupでも跨ぐ
- action-only overlapはWARN相当。次段でgroup-aware holdoutを設計する材料にします。


In [ ]:
print('Trajectory Leakage Gate:', summary['trajectory_leakage_gate'])
print('episodes hashed:', summary['episode_count_hashed'])
print('exact groups:', summary['exact_group_count'])
print('duplicate exact groups:', summary['duplicate_exact_group_count'])
print('episodes in duplicate exact groups:', summary['episodes_in_duplicate_exact_groups'])
print('action groups:', summary['action_group_count'])
print('duplicate action groups:', summary['duplicate_action_group_count'])
print('quality review crosscheck:', summary.get('quality_review_crosscheck'))

leaked = report[report['exact_leakage_group_count'] > 0]
if len(leaked):
    print('\nEXACT LEAKAGE DETECTED')
    display(leaked)
    members = pd.read_csv(LEAKAGE_OUT/'manifest_leakage_members.csv')
    display(members[members.leakage_kind == 'exact'].head(50))
else:
    print('\nNo exact trajectory-group leakage detected.')


## 出力

- `trajectory_group_leakage_summary.json`
- `manifest_leakage_report.csv`
- `manifest_leakage_members.csv`
- `trajectory_group_members.csv`
- `exact_trajectory_groups.csv`
- `action_trajectory_groups.csv`

**FAILなら `50_pi05_dataset_ablation.ipynb` はまだ走らせません。** 次にfixed evalをtrajectory-group-awareに作り直し、GateがPASSした後にcheap screeningへ進みます。
